# 02_parse_regulation_articles

## 목적

이 노트북은 `data/vectordb/`의 규정 PDF를 조문 단위 parent chunk로 변환한다.

01_check_pdf_extract.ipynb에서 확인한 결과:

- PDF 텍스트 추출 가능
- 조문 패턴 `제○조(제목)` 탐지 가능
- 전체 조문 후보 탐지 가능

이번 노트북에서 수행할 작업:

1. PDF별 페이지 텍스트 추출
2. 조문 헤더 `제○조(제목)` 위치 탐지
3. 조문 시작 위치부터 다음 조문 시작 전까지 본문 분리
4. article parent row 생성
5. `data/retrieval/parents.jsonl` 저장
6. 파싱 품질 진단 파일 저장

이번 노트북에서는 아직 child chunk를 만들지 않는다.
이번 노트북에서는 아직 ChromaDB, BM25 index를 만들지 않는다.

In [1]:
# 기본 라이브러리 및 경로 설정
from pathlib import Path
import re
import json
import hashlib
from pprint import pprint
from typing import List, Dict, Any, Optional
from collections import Counter, defaultdict

import pandas as pd

try:
    import fitz  # PyMuPDF
except ImportError:
    raise ImportError(
        "PyMuPDF가 설치되어 있지 않습니다. 아래 명령을 실행하세요:\n"
        "%pip install pymupdf"
    )


CURRENT_DIR = Path.cwd()

if CURRENT_DIR.name == "notebooks":
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    PROJECT_ROOT = CURRENT_DIR

PDF_DIR = PROJECT_ROOT / "data" / "vectordb"
RETRIEVAL_DIR = PROJECT_ROOT / "data" / "retrieval"
DEBUG_DIR = RETRIEVAL_DIR / "debug_article_parse"

PARENTS_PATH = RETRIEVAL_DIR / "parents.jsonl"
PARSED_ARTICLES_PATH = RETRIEVAL_DIR / "parsed_articles.jsonl"

DEBUG_DIR.mkdir(parents=True, exist_ok=True)
RETRIEVAL_DIR.mkdir(parents=True, exist_ok=True)

COLLECTION_NAME = "complypilot_regulations_v2"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("PDF_DIR:", PDF_DIR)
print("RETRIEVAL_DIR:", RETRIEVAL_DIR)
print("DEBUG_DIR:", DEBUG_DIR)
print("PARENTS_PATH:", PARENTS_PATH)
print("COLLECTION_NAME:", COLLECTION_NAME)

PROJECT_ROOT: c:\Users\USER\Desktop\complypilot-jb
PDF_DIR: c:\Users\USER\Desktop\complypilot-jb\data\vectordb
RETRIEVAL_DIR: c:\Users\USER\Desktop\complypilot-jb\data\retrieval
DEBUG_DIR: c:\Users\USER\Desktop\complypilot-jb\data\retrieval\debug_article_parse
PARENTS_PATH: c:\Users\USER\Desktop\complypilot-jb\data\retrieval\parents.jsonl
COLLECTION_NAME: complypilot_regulations_v2


In [2]:
# PDF 파일 목록 확인
pdf_files = sorted(PDF_DIR.glob("*.pdf"))

print("PDF 개수:", len(pdf_files))

for idx, pdf_path in enumerate(pdf_files, start=1):
    print(f"{idx:02d}. {pdf_path.name}")

PDF 개수: 8
01. 금융소비자 보호에 관한 감독규정(금융위원회고시)(제2026-11호)(20260402).pdf
02. 금융소비자 보호에 관한 법률 시행령(대통령령)(제36287호)(20260428).pdf
03. 금융소비자 보호에 관한 법률(법률)(제21065호)(20260102).pdf
04. 여신전문금융업감독규정(금융위원회고시)(제2026-17호)(20260506).pdf
05. 여신전문금융업법(법률)(제21065호)(20251001).pdf
06. 예금자보호법(법률)(제21065호)(20260102).pdf
07. 은행업감독규정 (금융위원회고시)(제2026-10호)(20260401).pdf
08. 표시ㆍ광고의 공정화에 관한 법률(법률)(제20712호)(20250121).pdf


In [3]:
# 문서 유형 및 문서 코드 추론 함수
def infer_document_type(file_name: str) -> str:
    """
    파일명을 기반으로 문서 유형을 추정합니다.

    Args:
        file_name: PDF 파일명

    Return:
        문서 유형 코드
    """
    name = file_name.replace(" ", "")

    if "시행령" in name:
        return "enforcement_decree"
    if "감독규정" in name:
        return "supervisory_regulation"
    if "가이드" in name or "해설" in name or "매뉴얼" in name:
        return "guideline"
    if "FAQ" in name or "질의" in name:
        return "faq_or_manual"
    if "법률" in name or name.endswith("법.pdf"):
        return "law"
    return "unknown"


def infer_document_priority(document_type: str) -> int:
    """
    문서 유형별 우선순위를 반환합니다.
    숫자가 낮을수록 상위 근거입니다.

    Args:
        document_type: 문서 유형 코드

    Return:
        문서 우선순위
    """
    priority_map = {
        "law": 1,
        "enforcement_decree": 2,
        "supervisory_regulation": 3,
        "guideline": 4,
        "faq_or_manual": 5,
        "unknown": 9,
    }
    return priority_map.get(document_type, 9)


def infer_doc_code(file_name: str) -> str:
    """
    파일명을 기반으로 사람이 읽을 수 있는 문서 코드를 생성합니다.

    Args:
        file_name: PDF 파일명

    Return:
        문서 코드
    """
    name = file_name.replace(" ", "")

    if "금융소비자" in name and "시행령" in name:
        return "financial_consumer_act_enforcement_decree"
    if "금융소비자" in name and "감독규정" in name:
        return "financial_consumer_supervisory_regulation"
    if "금융소비자" in name and "법률" in name:
        return "financial_consumer_act"
    if "여신전문금융업감독규정" in name:
        return "credit_finance_supervisory_regulation"
    if "여신전문금융업법" in name:
        return "credit_finance_business_act"
    if "예금자보호법" in name:
        return "depositor_protection_act"
    if "은행업감독규정" in name:
        return "banking_supervisory_regulation"
    if "표시" in name and "광고" in name:
        return "fair_labeling_advertising_act"

    digest = hashlib.md5(file_name.encode("utf-8")).hexdigest()[:8]
    return f"unknown_doc_{digest}"


def infer_law_name(file_name: str) -> str:
    """
    파일명에서 법령명을 추정합니다.

    Args:
        file_name: PDF 파일명

    Return:
        확장자를 제거한 법령명
    """
    return Path(file_name).stem.strip()


doc_meta_rows = []

for pdf_path in pdf_files:
    document_type = infer_document_type(pdf_path.name)
    doc_meta_rows.append({
        "file_name": pdf_path.name,
        "law_name": infer_law_name(pdf_path.name),
        "doc_code": infer_doc_code(pdf_path.name),
        "document_type": document_type,
        "document_priority": infer_document_priority(document_type),
    })

pd.DataFrame(doc_meta_rows)

,file_name,law_name,doc_code,document_type,document_priority
0,금융소비자 보호에 관한 감독규정(금융위원회고시)(제2026-11호)(20260402...,금융소비자 보호에 관한 감독규정(금융위원회고시)(제2026-11호)(20260402),financial_consumer_supervisory_regulation,supervisory_regulation,3
1,금융소비자 보호에 관한 법률 시행령(대통령령)(제36287호)(20260428).pdf,금융소비자 보호에 관한 법률 시행령(대통령령)(제36287호)(20260428),financial_consumer_act_enforcement_decree,enforcement_decree,2
2,금융소비자 보호에 관한 법률(법률)(제21065호)(20260102).pdf,금융소비자 보호에 관한 법률(법률)(제21065호)(20260102),financial_consumer_act,law,1
3,여신전문금융업감독규정(금융위원회고시)(제2026-17호)(20260506).pdf,여신전문금융업감독규정(금융위원회고시)(제2026-17호)(20260506),credit_finance_supervisory_regulation,supervisory_regulation,3
4,여신전문금융업법(법률)(제21065호)(20251001).pdf,여신전문금융업법(법률)(제21065호)(20251001),credit_finance_business_act,law,1
5,예금자보호법(법률)(제21065호)(20260102).pdf,예금자보호법(법률)(제21065호)(20260102),depositor_protection_act,law,1
6,은행업감독규정 (금융위원회고시)(제2026-10호)(20260401).pdf,은행업감독규정 (금융위원회고시)(제2026-10호)(20260401),banking_supervisory_regulation,supervisory_regulation,3
7,표시ㆍ광고의 공정화에 관한 법률(법률)(제20712호)(20250121).pdf,표시ㆍ광고의 공정화에 관한 법률(법률)(제20712호)(20250121),fair_labeling_advertising_act,law,1


In [4]:
# 페이지별 텍스트 추출 함수
def extract_page_texts(pdf_path: Path) -> List[Dict[str, Any]]:
    """
    PDF에서 페이지별 텍스트를 추출합니다.

    Args:
        pdf_path: PDF 파일 경로

    Return:
        page, text, text_length를 포함한 리스트
    """
    doc = fitz.open(pdf_path)
    page_texts = []

    for page_idx in range(len(doc)):
        text = doc[page_idx].get_text("text")

        page_texts.append({
            "page": page_idx + 1,
            "text": text,
            "text_length": len(text.strip()),
        })

    doc.close()
    return page_texts


sample_pdf = pdf_files[0]
sample_pages = extract_page_texts(sample_pdf)

print("샘플 PDF:", sample_pdf.name)
print("페이지 수:", len(sample_pages))
print("첫 페이지 텍스트 길이:", sample_pages[0]["text_length"])
print(sample_pages[0]["text"][:1500])

샘플 PDF: 금융소비자 보호에 관한 감독규정(금융위원회고시)(제2026-11호)(20260402).pdf
페이지 수: 28
첫 페이지 텍스트 길이: 1434
법제처                                                            1                                                   국가법령정보센터
금융소비자 보호에 관한 감독규정
금융소비자 보호에 관한 감독규정
[시행 2026. 4. 2.] [금융위원회고시 제2026-11호, 2026. 4. 2., 일부개정]
 
금융위원회(금융소비자정책과), 02-2100-2524
 
제1조(목적) 이 규정은 「금융소비자 보호에 관한 법률」 및 같은 법 시행령에서 위임하는 사항과 그 시행에 필요한
사항을 규정함을 목적으로 한다.
 
제2조(정의) ① 「금융소비자 보호에 관한 법률 시행령」(이하 "영"이라 한다) 제2조제1항제7호에서 "금융위원회가
정하여 고시하는 것"이란 다음 각 호의 어느 하나에 해당하는 것을 말한다.
1. 다음 각 목의 자가 계약에 따라 금융소비자로부터 금전을 받고 장래에 그 금전과 그에 따른 이자 등의 대가를
지급하기로 하는 계약. 다만, 「주택법」에 따른 입주자저축은 제외한다.
가. 「금융산업의 구조개선에 관한 법률」에 따라 「자본시장과 금융투자업에 관한 법률」에 따른 종합금융회사
와 합병한 기관(「예금자보호법」 제2조제1호가목부터 사목까지의 부보금융회사를 말한다)
나. 「농업협동조합법」에 따른 농협은행
다. 「상호저축은행법」에 따른 상호저축은행
라. 「수산업협동조합법」에 따른 수협은행
마. 「신용협동조합법」에 따른 조합(이하 "신용협동조합"이라 한다)
바. 「은행법」에 따라 인가를 받은 은행
사. 「자본시장과 금융투자업에 관한 법률」에 따른 금융투자업자 및 증권금융회사
아. 「자본시장과 금융투자업에 관한 법률」에 따른 종합금융회사
자. 「중소기업은행법」에 따른 중소기업은행
차. 「한국산업은행법」에 따른 한국산업은행
2. 

In [5]:
# 조문 헤더 정규식 정의
ARTICLE_HEADER_PATTERN = re.compile(
    r"제\s*\d+\s*조(?:의\s*\d+)?\s*\([^)]*\)"
)

ARTICLE_PARSE_PATTERN = re.compile(
    r"(제\s*\d+\s*조(?:의\s*\d+)?)\s*\(([^)]*)\)"
)


def normalize_spaces(text: str) -> str:
    """
    텍스트의 불필요한 공백을 정리합니다.
    줄 단위 구조는 최대한 유지합니다.

    Args:
        text: 원본 텍스트

    Return:
        정리된 텍스트
    """
    text = text.replace("\x00", " ")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


def parse_article_header(header: str) -> Dict[str, str]:
    """
    조문 헤더에서 조문번호와 조문제목을 분리합니다.

    Args:
        header: 예) 제17조(설명의무)

    Return:
        article_no, article_title, raw_header
    """
    match = ARTICLE_PARSE_PATTERN.search(header)

    if not match:
        return {
            "article_no": "",
            "article_title": "",
            "raw_header": header,
        }

    article_no = re.sub(r"\s+", "", match.group(1))
    article_title = match.group(2).strip()

    return {
        "article_no": article_no,
        "article_title": article_title,
        "raw_header": header,
    }


test_headers = [
    "제1조(목적)",
    "제 2 조(정의)",
    "제12조의2(광고의 제한)",
]

for header in test_headers:
    print(parse_article_header(header))

{'article_no': '제1조', 'article_title': '목적', 'raw_header': '제1조(목적)'}
{'article_no': '제2조', 'article_title': '정의', 'raw_header': '제 2 조(정의)'}
{'article_no': '제12조의2', 'article_title': '광고의 제한', 'raw_header': '제12조의2(광고의 제한)'}


In [6]:
# PDF 텍스트를 하나의 문자열로 합치고 page offset 만들기
def build_combined_text_with_offsets(page_texts: List[Dict[str, Any]]) -> Dict[str, Any]:
    """
    페이지별 텍스트를 하나의 문자열로 합치고, 각 페이지의 offset 범위를 기록합니다.

    Args:
        page_texts: 페이지별 텍스트 리스트

    Return:
        combined_text, page_offsets
    """
    combined_parts = []
    page_offsets = []

    current_offset = 0

    for page_item in page_texts:
        page = page_item["page"]
        text = normalize_spaces(page_item["text"])

        start_offset = current_offset
        combined_parts.append(text)

        current_offset += len(text)
        end_offset = current_offset

        page_offsets.append({
            "page": page,
            "start_offset": start_offset,
            "end_offset": end_offset,
        })

        page_break = f"\n\n[PAGE_BREAK:{page}]\n\n"
        combined_parts.append(page_break)
        current_offset += len(page_break)

    combined_text = "".join(combined_parts)

    return {
        "combined_text": combined_text,
        "page_offsets": page_offsets,
    }


def offset_to_page(offset: int, page_offsets: List[Dict[str, int]]) -> int:
    """
    combined_text의 offset이 어느 페이지에 속하는지 찾습니다.

    Args:
        offset: combined_text 기준 문자 위치
        page_offsets: 페이지별 offset 범위

    Return:
        페이지 번호
    """
    if not page_offsets:
        return 0

    for item in page_offsets:
        if item["start_offset"] <= offset <= item["end_offset"]:
            return item["page"]

    if offset < page_offsets[0]["start_offset"]:
        return page_offsets[0]["page"]

    return page_offsets[-1]["page"]


combined_info = build_combined_text_with_offsets(sample_pages)
combined_text = combined_info["combined_text"]
page_offsets = combined_info["page_offsets"]

print("combined_text 길이:", len(combined_text))
print("page_offsets 샘플:")
pprint(page_offsets[:3])
print(combined_text[:1000])

combined_text 길이: 39618
page_offsets 샘플:
[{'end_offset': 1325, 'page': 1, 'start_offset': 0},
 {'end_offset': 2860, 'page': 2, 'start_offset': 1343},
 {'end_offset': 4120, 'page': 3, 'start_offset': 2878}]
법제처 1 국가법령정보센터
금융소비자 보호에 관한 감독규정
금융소비자 보호에 관한 감독규정
[시행 2026. 4. 2.] [금융위원회고시 제2026-11호, 2026. 4. 2., 일부개정]
 
금융위원회(금융소비자정책과), 02-2100-2524
 
제1조(목적) 이 규정은 「금융소비자 보호에 관한 법률」 및 같은 법 시행령에서 위임하는 사항과 그 시행에 필요한
사항을 규정함을 목적으로 한다.
 
제2조(정의) ① 「금융소비자 보호에 관한 법률 시행령」(이하 "영"이라 한다) 제2조제1항제7호에서 "금융위원회가
정하여 고시하는 것"이란 다음 각 호의 어느 하나에 해당하는 것을 말한다.
1. 다음 각 목의 자가 계약에 따라 금융소비자로부터 금전을 받고 장래에 그 금전과 그에 따른 이자 등의 대가를
지급하기로 하는 계약. 다만, 「주택법」에 따른 입주자저축은 제외한다.
가. 「금융산업의 구조개선에 관한 법률」에 따라 「자본시장과 금융투자업에 관한 법률」에 따른 종합금융회사
와 합병한 기관(「예금자보호법」 제2조제1호가목부터 사목까지의 부보금융회사를 말한다)
나. 「농업협동조합법」에 따른 농협은행
다. 「상호저축은행법」에 따른 상호저축은행
라. 「수산업협동조합법」에 따른 수협은행
마. 「신용협동조합법」에 따른 조합(이하 "신용협동조합"이라 한다)
바. 「은행법」에 따라 인가를 받은 은행
사. 「자본시장과 금융투자업에 관한 법률」에 따른 금융투자업자 및 증권금융회사
아. 「자본시장과 금융투자업에 관한 법률」에 따른 종합금융회사
자. 「중소기업은행법」에 따른 중소기업은행
차. 「한국산업은행법」에 따른 한국

In [7]:
# 조문 헤더 후보 위치 수집
def find_article_header_candidates(
    combined_text: str,
    page_offsets: List[Dict[str, int]],
) -> List[Dict[str, Any]]:
    """
    combined_text에서 조문 헤더 후보와 위치를 수집합니다.

    Args:
        combined_text: PDF 전체 텍스트
        page_offsets: 페이지별 offset 범위

    Return:
        조문 헤더 후보 리스트
    """
    candidates = []

    for match in ARTICLE_HEADER_PATTERN.finditer(combined_text):
        raw_header = match.group(0)
        parsed = parse_article_header(raw_header)

        if not parsed["article_no"]:
            continue

        start = match.start()
        end = match.end()
        page = offset_to_page(start, page_offsets)

        candidates.append({
            "raw_header": raw_header,
            "article_no": parsed["article_no"],
            "article_title": parsed["article_title"],
            "start": start,
            "end": end,
            "page": page,
        })

    return candidates


sample_candidates = find_article_header_candidates(combined_text, page_offsets)

print("샘플 조문 후보 수:", len(sample_candidates))
pprint(sample_candidates[:20])

샘플 조문 후보 수: 38
[{'article_no': '제1조',
  'article_title': '목적',
  'end': 148,
  'page': 1,
  'raw_header': '제1조(목적)',
  'start': 141},
 {'article_no': '제2조',
  'article_title': '정의',
  'end': 232,
  'page': 1,
  'raw_header': '제2조(정의)',
  'start': 225},
 {'article_no': '제3조',
  'article_title': '금융상품의 유형',
  'end': 3703,
  'page': 3,
  'raw_header': '제3조(금융상품의 유형)',
  'start': 3690},
 {'article_no': '제4조',
  'article_title': '금융회사등의 업종구분',
  'end': 4187,
  'page': 4,
  'raw_header': '제4조(금융회사등의 업종구분)',
  'start': 4171},
 {'article_no': '제5조',
  'article_title': '금융상품자문업자의 등록요건',
  'end': 4295,
  'page': 4,
  'raw_header': '제5조(금융상품자문업자의 등록요건)',
  'start': 4276},
 {'article_no': '제6조',
  'article_title': '금융상품판매대리ㆍ중개업자의 등록요건',
  'end': 5777,
  'page': 5,
  'raw_header': '제6조(금융상품판매대리ㆍ중개업자의 등록요건)',
  'start': 5753},
 {'article_no': '제7조',
  'article_title': '등록신청',
  'end': 7525,
  'page': 6,
  'raw_header': '제7조(등록신청)',
  'start': 7516},
 {'article_no': '제8조',
  'article_title': '등록수수료',

In [8]:
# 조문 본문 분리 함수
def make_parent_id(doc_code: str, article_no: str) -> str:
    """
    문서 코드와 조문번호로 parent_id를 생성합니다.

    Args:
        doc_code: 문서 코드
        article_no: 조문번호

    Return:
        parent_id
    """
    article_key = (
        article_no
        .replace("제", "article_")
        .replace("조의", "_")
        .replace("조", "")
    )
    article_key = re.sub(r"[^a-zA-Z0-9_]+", "_", article_key)
    article_key = re.sub(r"_+", "_", article_key).strip("_")
    return f"{doc_code}__{article_key}"


def extract_effective_date_from_text(text: str) -> str:
    """
    문서 앞부분에서 시행일 정보를 추정합니다.

    Args:
        text: 문서 전체 또는 앞부분 텍스트

    Return:
        시행일 문자열. 찾지 못하면 빈 문자열
    """
    patterns = [
        r"\[시행\s*([0-9]{4}\.\s*[0-9]{1,2}\.\s*[0-9]{1,2}\.?)\]",
        r"시행\s*([0-9]{4}\.\s*[0-9]{1,2}\.\s*[0-9]{1,2}\.?)",
        r"시행일\s*[:：]?\s*([0-9]{4}\.\s*[0-9]{1,2}\.\s*[0-9]{1,2}\.?)",
    ]

    head = text[:5000]

    for pattern in patterns:
        match = re.search(pattern, head)
        if match:
            return re.sub(r"\s+", "", match.group(1)).strip()

    return ""


def segment_articles_from_pdf(pdf_path: Path) -> List[Dict[str, Any]]:
    """
    PDF를 조문 단위 parent article로 분리합니다.

    Args:
        pdf_path: PDF 파일 경로

    Return:
        조문 parent row 리스트
    """
    page_texts = extract_page_texts(pdf_path)
    combined_info = build_combined_text_with_offsets(page_texts)

    combined_text = combined_info["combined_text"]
    page_offsets = combined_info["page_offsets"]

    candidates = find_article_header_candidates(combined_text, page_offsets)

    file_name = pdf_path.name
    law_name = infer_law_name(file_name)
    document_type = infer_document_type(file_name)
    document_priority = infer_document_priority(document_type)
    doc_code = infer_doc_code(file_name)
    effective_date = extract_effective_date_from_text(combined_text)

    articles = []

    for idx, candidate in enumerate(candidates):
        start = candidate["start"]
        next_start = candidates[idx + 1]["start"] if idx + 1 < len(candidates) else len(combined_text)

        article_text = combined_text[start:next_start]
        article_text = re.sub(r"\[PAGE_BREAK:\d+\]", "", article_text)
        article_text = normalize_spaces(article_text)

        page_start = candidate["page"]
        page_end = offset_to_page(next_start, page_offsets)

        article_no = candidate["article_no"]
        article_title = candidate["article_title"]
        parent_id = make_parent_id(doc_code, article_no)

        articles.append({
            "parent_id": parent_id,
            "doc_code": doc_code,
            "law_name": law_name,
            "document_type": document_type,
            "document_priority": document_priority,
            "article_no": article_no,
            "article_title": article_title,
            "page_start": page_start,
            "page_end": page_end,
            "effective_date": effective_date,
            "source_file": file_name,
            "text": article_text,
            "text_length": len(article_text),
        })

    return articles


sample_articles = segment_articles_from_pdf(sample_pdf)

print("샘플 PDF:", sample_pdf.name)
print("분리된 article 수:", len(sample_articles))
pprint(sample_articles[:3])

샘플 PDF: 금융소비자 보호에 관한 감독규정(금융위원회고시)(제2026-11호)(20260402).pdf
분리된 article 수: 38
[{'article_no': '제1조',
  'article_title': '목적',
  'doc_code': 'financial_consumer_supervisory_regulation',
  'document_priority': 3,
  'document_type': 'supervisory_regulation',
  'effective_date': '2026.4.2.',
  'law_name': '금융소비자 보호에 관한 감독규정(금융위원회고시)(제2026-11호)(20260402)',
  'page_end': 1,
  'page_start': 1,
  'parent_id': 'financial_consumer_supervisory_regulation__article_1',
  'source_file': '금융소비자 보호에 관한 감독규정(금융위원회고시)(제2026-11호)(20260402).pdf',
  'text': '제1조(목적) 이 규정은 「금융소비자 보호에 관한 법률」 및 같은 법 시행령에서 위임하는 사항과 그 시행에 필요한\n'
          '사항을 규정함을 목적으로 한다.',
  'text_length': 81},
 {'article_no': '제2조',
  'article_title': '정의',
  'doc_code': 'financial_consumer_supervisory_regulation',
  'document_priority': 3,
  'document_type': 'supervisory_regulation',
  'effective_date': '2026.4.2.',
  'law_name': '금융소비자 보호에 관한 감독규정(금융위원회고시)(제2026-11호)(20260402)',
  'page_end': 3,
  'page_start': 1,
  'parent_id': 'financia

In [9]:
# 샘플 article 본문 확인
for idx, article in enumerate(sample_articles[:5], start=1):
    print("=" * 120)
    print(f"[{idx}] {article['law_name']} {article['article_no']}({article['article_title']})")
    print(f"page: {article['page_start']}~{article['page_end']}")
    print(f"parent_id: {article['parent_id']}")
    print(f"text_length: {article['text_length']}")
    print("-" * 120)
    print(article["text"][:2000])

[1] 금융소비자 보호에 관한 감독규정(금융위원회고시)(제2026-11호)(20260402) 제1조(목적)
page: 1~1
parent_id: financial_consumer_supervisory_regulation__article_1
text_length: 81
------------------------------------------------------------------------------------------------------------------------
제1조(목적) 이 규정은 「금융소비자 보호에 관한 법률」 및 같은 법 시행령에서 위임하는 사항과 그 시행에 필요한
사항을 규정함을 목적으로 한다.
[2] 금융소비자 보호에 관한 감독규정(금융위원회고시)(제2026-11호)(20260402) 제2조(정의)
page: 1~3
parent_id: financial_consumer_supervisory_regulation__article_2
text_length: 3430
------------------------------------------------------------------------------------------------------------------------
제2조(정의) ① 「금융소비자 보호에 관한 법률 시행령」(이하 "영"이라 한다) 제2조제1항제7호에서 "금융위원회가
정하여 고시하는 것"이란 다음 각 호의 어느 하나에 해당하는 것을 말한다.
1. 다음 각 목의 자가 계약에 따라 금융소비자로부터 금전을 받고 장래에 그 금전과 그에 따른 이자 등의 대가를
지급하기로 하는 계약. 다만, 「주택법」에 따른 입주자저축은 제외한다.
가. 「금융산업의 구조개선에 관한 법률」에 따라 「자본시장과 금융투자업에 관한 법률」에 따른 종합금융회사
와 합병한 기관(「예금자보호법」 제2조제1호가목부터 사목까지의 부보금융회사를 말한다)
나. 「농업협동조합법」에 따른 농협은행
다. 「상호저축은행법」에 따른 상호저축은행
라. 「수산업협동조합

In [10]:
# 너무 짧은 조문 후보 진단
def diagnose_articles(articles: List[Dict[str, Any]]) -> Dict[str, Any]:
    """
    article 리스트의 품질을 진단합니다.

    Args:
        articles: 조문 parent row 리스트

    Return:
        진단 요약 dict
    """
    text_lengths = [row["text_length"] for row in articles]

    if not articles:
        return {
            "article_count": 0,
            "short_article_count_lt_50": 0,
            "short_article_count_lt_100": 0,
            "avg_text_length": 0,
            "min_text_length": 0,
            "max_text_length": 0,
        }

    return {
        "article_count": len(articles),
        "short_article_count_lt_50": sum(1 for x in text_lengths if x < 50),
        "short_article_count_lt_100": sum(1 for x in text_lengths if x < 100),
        "avg_text_length": round(sum(text_lengths) / len(text_lengths), 2),
        "min_text_length": min(text_lengths),
        "max_text_length": max(text_lengths),
    }


sample_diagnosis = diagnose_articles(sample_articles)
pprint(sample_diagnosis)

short_articles = [row for row in sample_articles if row["text_length"] < 100]

print("짧은 article 샘플 수:", len(short_articles))
for row in short_articles[:10]:
    print(row["article_no"], row["article_title"], row["page_start"], row["text_length"], row["text"][:200])

{'article_count': 38,
 'avg_text_length': 1023.92,
 'max_text_length': 5091,
 'min_text_length': 75,
 'short_article_count_lt_100': 4,
 'short_article_count_lt_50': 0}
짧은 article 샘플 수: 4
제1조 목적 1 81 제1조(목적) 이 규정은 「금융소비자 보호에 관한 법률」 및 같은 법 시행령에서 위임하는 사항과 그 시행에 필요한
사항을 규정함을 목적으로 한다.
제34조의2 과징금의 부과기준 28 90 제34조의2(과징금의 부과기준) 영 제43조 제1항ㆍ제2항 및 영 별표 3에 따른 과징금의 부과기준은 별표 7과 같이
한다.
[본조신설 2025. 11. 19.]
제1조 시행일 28 75 제1조(시행일) 이 규정은 고시한 날부터 시행한다. 다만, 제13조제1항제5호 개정 규정은 고시 후 3개월이 경과한 날
부터 시행한다.
제2조 적용례 28 80 제2조(적용례) 제13조제1항제5호의 개정 규정은 이 규정 시행 이후 금융상품판매업자가 일반금융소비자에게 설명
서를 제공하는 경우부터 적용한다.


In [11]:
# 전체 PDF article 파싱 실행
all_articles = []
parse_errors = []

for pdf_path in pdf_files:
    try:
        articles = segment_articles_from_pdf(pdf_path)
        all_articles.extend(articles)

        print(
            f"[OK] {pdf_path.name} | articles: {len(articles)}"
        )
    except Exception as e:
        parse_errors.append({
            "file_name": pdf_path.name,
            "error": str(e),
        })
        print(f"[ERROR] {pdf_path.name} | {e}")

print("=" * 100)
print("전체 article 수:", len(all_articles))
print("파싱 에러 수:", len(parse_errors))

if parse_errors:
    pprint(parse_errors)

[OK] 금융소비자 보호에 관한 감독규정(금융위원회고시)(제2026-11호)(20260402).pdf | articles: 38
[OK] 금융소비자 보호에 관한 법률 시행령(대통령령)(제36287호)(20260428).pdf | articles: 53
[OK] 금융소비자 보호에 관한 법률(법률)(제21065호)(20260102).pdf | articles: 75
[OK] 여신전문금융업감독규정(금융위원회고시)(제2026-17호)(20260506).pdf | articles: 88
[OK] 여신전문금융업법(법률)(제21065호)(20251001).pdf | articles: 112
[OK] 예금자보호법(법률)(제21065호)(20260102).pdf | articles: 86
[OK] 은행업감독규정 (금융위원회고시)(제2026-10호)(20260401).pdf | articles: 132
[OK] 표시ㆍ광고의 공정화에 관한 법률(법률)(제20712호)(20250121).pdf | articles: 24
전체 article 수: 608
파싱 에러 수: 0


In [12]:
# 전체 파싱 결과 요약
df_articles = pd.DataFrame(all_articles)

print("전체 row 수:", len(df_articles))

if not df_articles.empty:
    display_cols = [
        "law_name",
        "document_type",
        "article_no",
        "article_title",
        "page_start",
        "page_end",
        "text_length",
        "parent_id",
    ]

    display(df_articles[display_cols].head(30))

    print("문서별 article 수:")
    display(df_articles.groupby(["law_name", "document_type"]).size().reset_index(name="article_count"))

    print("문서 유형별 article 수:")
    display(df_articles["document_type"].value_counts().reset_index())

전체 row 수: 608


,law_name,document_type,article_no,article_title,page_start,page_end,text_length,parent_id
0,금융소비자 보호에 관한 감독규정(금융위원회고시)(제2026-11호)(20260402),supervisory_regulation,제1조,목적,1,1,81,financial_consumer_supervisory_regulation__art...
1,금융소비자 보호에 관한 감독규정(금융위원회고시)(제2026-11호)(20260402),supervisory_regulation,제2조,정의,1,3,3430,financial_consumer_supervisory_regulation__art...
2,금융소비자 보호에 관한 감독규정(금융위원회고시)(제2026-11호)(20260402),supervisory_regulation,제3조,금융상품의 유형,3,4,464,financial_consumer_supervisory_regulation__art...
3,금융소비자 보호에 관한 감독규정(금융위원회고시)(제2026-11호)(20260402),supervisory_regulation,제4조,금융회사등의 업종구분,4,4,102,financial_consumer_supervisory_regulation__art...
4,금융소비자 보호에 관한 감독규정(금융위원회고시)(제2026-11호)(20260402),supervisory_regulation,제5조,금융상품자문업자의 등록요건,4,5,1460,financial_consumer_supervisory_regulation__art...
5,금융소비자 보호에 관한 감독규정(금융위원회고시)(제2026-11호)(20260402),supervisory_regulation,제6조,금융상품판매대리ㆍ중개업자의 등록요건,5,6,1744,financial_consumer_supervisory_regulation__art...
6,금융소비자 보호에 관한 감독규정(금융위원회고시)(제2026-11호)(20260402),supervisory_regulation,제7조,등록신청,6,6,1034,financial_consumer_supervisory_regulation__art...
7,금융소비자 보호에 관한 감독규정(금융위원회고시)(제2026-11호)(20260402),supervisory_regulation,제8조,등록수수료,6,7,153,financial_consumer_supervisory_regulation__art...
8,금융소비자 보호에 관한 감독규정(금융위원회고시)(제2026-11호)(20260402),supervisory_regulation,제9조,내부통제기준,7,7,889,financial_consumer_supervisory_regulation__art...
9,금융소비자 보호에 관한 감독규정(금융위원회고시)(제2026-11호)(20260402),supervisory_regulation,제10조,적합성 원칙,7,9,1861,financial_consumer_supervisory_regulation__art...


문서별 article 수:


,law_name,document_type,article_count
0,금융소비자 보호에 관한 감독규정(금융위원회고시)(제2026-11호)(20260402),supervisory_regulation,38
1,금융소비자 보호에 관한 법률 시행령(대통령령)(제36287호)(20260428),enforcement_decree,53
2,금융소비자 보호에 관한 법률(법률)(제21065호)(20260102),law,75
3,여신전문금융업감독규정(금융위원회고시)(제2026-17호)(20260506),supervisory_regulation,88
4,여신전문금융업법(법률)(제21065호)(20251001),law,112
5,예금자보호법(법률)(제21065호)(20260102),law,86
6,은행업감독규정 (금융위원회고시)(제2026-10호)(20260401),supervisory_regulation,132
7,표시ㆍ광고의 공정화에 관한 법률(법률)(제20712호)(20250121),law,24


문서 유형별 article 수:


,document_type,count
0,law,297
1,supervisory_regulation,258
2,enforcement_decree,53


In [13]:
# 중복 parent_id 진단
if not df_articles.empty:
    duplicate_parent_ids = (
        df_articles["parent_id"]
        .value_counts()
        .reset_index()
    )
    duplicate_parent_ids.columns = ["parent_id", "count"]

    duplicate_parent_ids = duplicate_parent_ids[duplicate_parent_ids["count"] > 1]

    print("중복 parent_id 수:", len(duplicate_parent_ids))

    if len(duplicate_parent_ids) > 0:
        display(duplicate_parent_ids.head(20))

        duplicated_samples = df_articles[df_articles["parent_id"].isin(duplicate_parent_ids["parent_id"].head(5))]
        display(duplicated_samples[[
            "law_name",
            "article_no",
            "article_title",
            "page_start",
            "page_end",
            "text_length",
            "parent_id",
        ]])

중복 parent_id 수: 13


,parent_id,count
0,credit_finance_business_act__article_46,3
1,financial_consumer_supervisory_regulation__art...,2
2,financial_consumer_supervisory_regulation__art...,2
3,financial_consumer_act__article_1,2
4,financial_consumer_act__article_7,2
5,credit_finance_business_act__article_1,2
6,credit_finance_business_act__article_7,2
7,credit_finance_business_act__article_24_2,2
8,depositor_protection_act__article_1,2
9,depositor_protection_act__article_7,2


,law_name,article_no,article_title,page_start,page_end,text_length,parent_id
0,금융소비자 보호에 관한 감독규정(금융위원회고시)(제2026-11호)(20260402),제1조,목적,1,1,81,financial_consumer_supervisory_regulation__art...
1,금융소비자 보호에 관한 감독규정(금융위원회고시)(제2026-11호)(20260402),제2조,정의,1,3,3430,financial_consumer_supervisory_regulation__art...
36,금융소비자 보호에 관한 감독규정(금융위원회고시)(제2026-11호)(20260402),제1조,시행일,28,28,75,financial_consumer_supervisory_regulation__art...
37,금융소비자 보호에 관한 감독규정(금융위원회고시)(제2026-11호)(20260402),제2조,적용례,28,28,80,financial_consumer_supervisory_regulation__art...
91,금융소비자 보호에 관한 법률(법률)(제21065호)(20260102),제1조,목적,1,1,191,financial_consumer_act__article_1
97,금융소비자 보호에 관한 법률(법률)(제21065호)(20260102),제7조,금융소비자의 기본적 권리,3,4,364,financial_consumer_act__article_7
164,금융소비자 보호에 관한 법률(법률)(제21065호)(20260102),제1조,시행일,25,25,518,financial_consumer_act__article_1
165,금융소비자 보호에 관한 법률(법률)(제21065호)(20260102),제7조,다른 법률의 개정,25,25,129,financial_consumer_act__article_7
317,여신전문금융업법(법률)(제21065호)(20251001),제46조,업무,17,18,790,credit_finance_business_act__article_46
344,여신전문금융업법(법률)(제21065호)(20251001),제46조,이 항 각 호\n외의 부분에서 정하는 업무에 관한 규정으로 한정한다,25,25,1159,credit_finance_business_act__article_46


In [14]:
# 저장 전 parent_id 충돌 방지
def make_unique_parent_ids(articles: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    """
    parent_id 중복을 방지하기 위해 중복 parent_id에 순번 suffix를 붙입니다.

    Args:
        articles: 조문 parent row 리스트

    Return:
        parent_id가 유니크해진 article 리스트
    """
    seen = defaultdict(int)
    updated = []

    for row in articles:
        row = dict(row)
        base_id = row["parent_id"]

        seen[base_id] += 1

        if seen[base_id] == 1:
            row["parent_id"] = base_id
        else:
            row["parent_id"] = f"{base_id}__dup_{seen[base_id]}"

        updated.append(row)

    return updated


all_articles_unique = make_unique_parent_ids(all_articles)

parent_id_counts = Counter(row["parent_id"] for row in all_articles_unique)
duplicated_after = [parent_id for parent_id, count in parent_id_counts.items() if count > 1]

print("유니크 처리 후 중복 parent_id 수:", len(duplicated_after))

유니크 처리 후 중복 parent_id 수: 0


In [15]:
# 너무 짧은 article 필터링 여부 판단
def add_quality_flags(articles: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    """
    article row에 품질 진단 플래그를 추가합니다.

    Args:
        articles: 조문 parent row 리스트

    Return:
        품질 플래그가 추가된 article 리스트
    """
    updated = []

    for row in articles:
        row = dict(row)
        text = row.get("text", "")
        text_length = len(text)

        row["is_short_article"] = text_length < 80
        row["has_article_header"] = bool(row.get("article_no")) and bool(row.get("article_title"))
        row["parse_status"] = "ok"

        if row["is_short_article"]:
            row["parse_status"] = "short_text_review"

        if not row["has_article_header"]:
            row["parse_status"] = "missing_header_review"

        updated.append(row)

    return updated


all_articles_final = add_quality_flags(all_articles_unique)

df_final = pd.DataFrame(all_articles_final)

print("최종 article 수:", len(df_final))
print("parse_status 분포:")
display(df_final["parse_status"].value_counts().reset_index())

display(df_final[[
    "law_name",
    "article_no",
    "article_title",
    "page_start",
    "page_end",
    "text_length",
    "parse_status",
]].head(30))

최종 article 수: 608
parse_status 분포:


,parse_status,count
0,ok,583
1,short_text_review,25


,law_name,article_no,article_title,page_start,page_end,text_length,parse_status
0,금융소비자 보호에 관한 감독규정(금융위원회고시)(제2026-11호)(20260402),제1조,목적,1,1,81,ok
1,금융소비자 보호에 관한 감독규정(금융위원회고시)(제2026-11호)(20260402),제2조,정의,1,3,3430,ok
2,금융소비자 보호에 관한 감독규정(금융위원회고시)(제2026-11호)(20260402),제3조,금융상품의 유형,3,4,464,ok
3,금융소비자 보호에 관한 감독규정(금융위원회고시)(제2026-11호)(20260402),제4조,금융회사등의 업종구분,4,4,102,ok
4,금융소비자 보호에 관한 감독규정(금융위원회고시)(제2026-11호)(20260402),제5조,금융상품자문업자의 등록요건,4,5,1460,ok
5,금융소비자 보호에 관한 감독규정(금융위원회고시)(제2026-11호)(20260402),제6조,금융상품판매대리ㆍ중개업자의 등록요건,5,6,1744,ok
6,금융소비자 보호에 관한 감독규정(금융위원회고시)(제2026-11호)(20260402),제7조,등록신청,6,6,1034,ok
7,금융소비자 보호에 관한 감독규정(금융위원회고시)(제2026-11호)(20260402),제8조,등록수수료,6,7,153,ok
8,금융소비자 보호에 관한 감독규정(금융위원회고시)(제2026-11호)(20260402),제9조,내부통제기준,7,7,889,ok
9,금융소비자 보호에 관한 감독규정(금융위원회고시)(제2026-11호)(20260402),제10조,적합성 원칙,7,9,1861,ok


In [16]:
# JSONL 저장 함수
def save_jsonl(rows: List[Dict[str, Any]], path: Path) -> None:
    """
    dict 리스트를 JSONL 파일로 저장합니다.

    Args:
        rows: 저장할 dict 리스트
        path: 저장 경로

    Return:
        None
    """
    path.parent.mkdir(parents=True, exist_ok=True)

    with open(path, "w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")


save_jsonl(all_articles_final, PARENTS_PATH)
save_jsonl(all_articles_final, PARSED_ARTICLES_PATH)

print("parents.jsonl 저장 완료:", PARENTS_PATH)
print("parsed_articles.jsonl 저장 완료:", PARSED_ARTICLES_PATH)

parents.jsonl 저장 완료: c:\Users\USER\Desktop\complypilot-jb\data\retrieval\parents.jsonl
parsed_articles.jsonl 저장 완료: c:\Users\USER\Desktop\complypilot-jb\data\retrieval\parsed_articles.jsonl


In [17]:
# 저장 파일 재로드 검증
def load_jsonl(path: Path) -> List[Dict[str, Any]]:
    """
    JSONL 파일을 다시 읽습니다.

    Args:
        path: JSONL 파일 경로

    Return:
        dict 리스트
    """
    rows = []

    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                rows.append(json.loads(line))

    return rows


reloaded_parents = load_jsonl(PARENTS_PATH)

print("재로드 row 수:", len(reloaded_parents))
print("첫 row:")
pprint(reloaded_parents[0] if reloaded_parents else None)

assert len(reloaded_parents) == len(all_articles_final), "저장 row 수와 재로드 row 수가 다릅니다."
assert all("parent_id" in row for row in reloaded_parents), "parent_id 누락 row가 있습니다."
assert all("article_no" in row for row in reloaded_parents), "article_no 누락 row가 있습니다."

print("[OK] parents.jsonl 재로드 검증 완료")

재로드 row 수: 608
첫 row:
{'article_no': '제1조',
 'article_title': '목적',
 'doc_code': 'financial_consumer_supervisory_regulation',
 'document_priority': 3,
 'document_type': 'supervisory_regulation',
 'effective_date': '2026.4.2.',
 'has_article_header': True,
 'is_short_article': False,
 'law_name': '금융소비자 보호에 관한 감독규정(금융위원회고시)(제2026-11호)(20260402)',
 'page_end': 1,
 'page_start': 1,
 'parent_id': 'financial_consumer_supervisory_regulation__article_1',
 'parse_status': 'ok',
 'source_file': '금융소비자 보호에 관한 감독규정(금융위원회고시)(제2026-11호)(20260402).pdf',
 'text': '제1조(목적) 이 규정은 「금융소비자 보호에 관한 법률」 및 같은 법 시행령에서 위임하는 사항과 그 시행에 필요한\n'
         '사항을 규정함을 목적으로 한다.',
 'text_length': 81}
[OK] parents.jsonl 재로드 검증 완료


In [18]:
# 디버그 요약 저장
parse_summary = {
    "pdf_count": len(pdf_files),
    "article_count": len(all_articles_final),
    "parse_error_count": len(parse_errors),
    "parents_path": str(PARENTS_PATH),
    "parsed_articles_path": str(PARSED_ARTICLES_PATH),
    "status_counts": df_final["parse_status"].value_counts().to_dict() if not df_final.empty else {},
    "document_type_counts": df_final["document_type"].value_counts().to_dict() if not df_final.empty else {},
    "article_count_by_law": (
        df_final.groupby("law_name").size().to_dict()
        if not df_final.empty
        else {}
    ),
    "parse_errors": parse_errors,
}

summary_path = DEBUG_DIR / "article_parse_summary.json"

with open(summary_path, "w", encoding="utf-8") as f:
    json.dump(parse_summary, f, ensure_ascii=False, indent=2)

print("파싱 요약 저장:", summary_path)
pprint(parse_summary)

파싱 요약 저장: c:\Users\USER\Desktop\complypilot-jb\data\retrieval\debug_article_parse\article_parse_summary.json
{'article_count': 608,
 'article_count_by_law': {'금융소비자 보호에 관한 감독규정(금융위원회고시)(제2026-11호)(20260402)': 38,
                          '금융소비자 보호에 관한 법률 시행령(대통령령)(제36287호)(20260428)': 53,
                          '금융소비자 보호에 관한 법률(법률)(제21065호)(20260102)': 75,
                          '여신전문금융업감독규정(금융위원회고시)(제2026-17호)(20260506)': 88,
                          '여신전문금융업법(법률)(제21065호)(20251001)': 112,
                          '예금자보호법(법률)(제21065호)(20260102)': 86,
                          '은행업감독규정 (금융위원회고시)(제2026-10호)(20260401)': 132,
                          '표시ㆍ광고의 공정화에 관한 법률(법률)(제20712호)(20250121)': 24},
 'document_type_counts': {'enforcement_decree': 53,
                          'law': 297,
                          'supervisory_regulation': 258},
 'parents_path': 'c:\\Users\\USER\\Desktop\\complypilot-jb\\data\\retrieval\\parents.jsonl',
 'parse_error_count': 0,
 'parse_errors': []

In [19]:
# 최종 체크
print("=" * 100)
print("02_parse_regulation_articles 최종 체크")
print("=" * 100)

checks = {
    "pdf_count_gt_0": len(pdf_files) > 0,
    "article_count_gt_0": len(all_articles_final) > 0,
    "parents_jsonl_exists": PARENTS_PATH.exists(),
    "parents_reload_match": len(reloaded_parents) == len(all_articles_final),
    "no_parse_errors": len(parse_errors) == 0,
    "has_page_metadata": all(
        "page_start" in row and "page_end" in row
        for row in all_articles_final
    ),
    "has_document_type": all(
        "document_type" in row
        for row in all_articles_final
    ),
}

pprint(checks)

if all(checks.values()):
    print("[OK] 조문 단위 parent artifact 생성 완료")
else:
    print("[WARN] 일부 체크가 실패했습니다. 위 결과를 보고 보완이 필요합니다.")

02_parse_regulation_articles 최종 체크
{'article_count_gt_0': True,
 'has_document_type': True,
 'has_page_metadata': True,
 'no_parse_errors': True,
 'parents_jsonl_exists': True,
 'parents_reload_match': True,
 'pdf_count_gt_0': True}
[OK] 조문 단위 parent artifact 생성 완료
